In [ ]:
!pip install spotipy

import pandas as pd
from tqdm import tqdm
import spotipy # Changed import
from spotipy.oauth2 import SpotifyClientCredentials # Changed import
from datetime import datetime, timedelta

# -----------------------------
# 1️⃣ Spotify Authentication
# -----------------------------
CLIENT_ID = ""
CLIENT_SECRET = ""

sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET
))

# -----------------------------
# 2️⃣ Parameters
# -----------------------------
RECENT_MONTHS = 3  # look at tracks released in last 3 months
recent_date_threshold = datetime.now() - timedelta(days=30*RECENT_MONTHS)

# -----------------------------
# 3️⃣ Discover Artists & Genres
# -----------------------------
artist_sample = []
all_genres_set = set()

# Sample artists by searching common letters to get broad coverage
for char in tqdm("abcdefghijklmnopqrstuvwxyz", desc="Searching artists"):
    results = sp.search(q=char, type='artist', limit=50)
    artists = results.get('artists', {}).get('items', [])

    for artist in artists:
        artist_sample.append(artist)
        for genre in artist.get('genres', []):
            all_genres_set.add(genre)

all_genres = sorted(list(all_genres_set))
print(f"Discovered {len(all_genres)} unique Spotify genres.")

# -----------------------------
# 4️⃣ Compute Genre-Level Metrics
# -----------------------------
genre_metrics = []

for genre in tqdm(all_genres, desc="Processing genres"):
    # Filter artists for this genre
    artists_in_genre = [a for a in artist_sample if genre in a.get('genres', [])]

    if not artists_in_genre:
        continue

    num_artists = len(artists_in_genre)
    avg_artist_popularity = sum(a['popularity'] for a in artists_in_genre) / num_artists
    avg_followers = sum(a['followers']['total'] for a in artists_in_genre) / num_artists

    # Top artist
    top_artist = max(artists_in_genre, key=lambda x: x['followers']['total'])
    top_artist_name = top_artist['name']
    top_artist_followers = top_artist['followers']['total']

    # Track-level metrics for recent growth
    recent_tracks_count = 0
    recent_tracks_popularity = []

    for artist in artists_in_genre:
        artist_id = artist['id']
        top_tracks = sp.artist_top_tracks(artist_id, country='US')['tracks']

        for track in top_tracks:
            release_date_str = track.get('album', {}).get('release_date', '')
            release_precision = track.get('album', {}).get('release_date_precision', '')

            # Convert release_date to datetime
            try:
                if release_precision == 'day':
                    release_date = datetime.strptime(release_date_str, '%Y-%m-%d')
                elif release_precision == 'month':
                    release_date = datetime.strptime(release_date_str, '%Y-%m')
                elif release_precision == 'year':
                    release_date = datetime.strptime(release_date_str, '%Y')
                else:
                    continue
            except:
                continue

            if release_date >= recent_date_threshold:
                recent_tracks_count += 1
                recent_tracks_popularity.append(track['popularity'])

    avg_recent_track_popularity = (
        sum(recent_tracks_popularity) / len(recent_tracks_popularity)
        if recent_tracks_popularity else 0
    )

    genre_metrics.append({
        "genre": genre,
        "num_artists": num_artists,
        "avg_artist_popularity": avg_artist_popularity,
        "avg_followers": avg_followers,
        "top_artist_name": top_artist_name,
        "top_artist_followers": top_artist_followers,
        "num_tracks_last_3mo": recent_tracks_count,
        "avg_track_popularity": avg_recent_track_popularity
    })

# -----------------------------
# 5️⃣ Save Final CSV
# -----------------------------
df_genre_metrics = pd.DataFrame(genre_metrics)
df_genre_metrics.to_csv("spotify_genres_growth.csv", index=False)
print("Saved final genre-level dataset to spotify_genres_growth.csv")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 279.8/279.8 kB 7.8 MB/s eta 0:00:00


Searching artists: 100%|██████████| 26/26 [00:12<00:00,  2.16it/s]


Discovered 334 unique Spotify genres.


Processing genres: 100%|██████████| 334/334 [06:20<00:00,  1.14s/it]

Saved final genre-level dataset to spotify_genres_growth.csv


In [ ]:
from google.colab import files

files.download("spotify_genres_growth.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
df = pd.read_csv("spotify_genres_growth.csv")

# Normalize columns
df['norm_avg_pop'] = df['avg_artist_popularity'] / 100
df['norm_avg_followers'] = df['avg_followers'] / df['avg_followers'].max()

# Avoid division by zero for recent tracks
max_recent_tracks = df['num_tracks_last_3mo'].max()
if max_recent_tracks == 0:
    max_recent_tracks = 1

df['norm_recent_tracks'] = df['num_tracks_last_3mo'] / max_recent_tracks
df['norm_recent_track_pop'] = df['avg_track_popularity'] / 100

# Growth score formula
df['growth_score'] = (
    0.3 * df['norm_recent_tracks'] +       # recent activity
    0.3 * df['norm_recent_track_pop'] +    # popularity of recent tracks
    0.2 * df['norm_avg_pop'] +             # average artist popularity
    0.2 * (1 - df['norm_avg_followers'])   # low saturation favored
)

# Sort by growth score descending
df_sorted = df.sort_values('growth_score', ascending=False)

# Display top 20 emerging genres
print(df_sorted[['genre', 'growth_score', 'num_artists', 'avg_artist_popularity', 'avg_followers', 'num_tracks_last_3mo', 'avg_track_popularity']].head(20))

                   genre  growth_score  num_artists  avg_artist_popularity  \
177                k-pop      0.864280           46              83.130435   
82               country      0.823219           71              75.957746   
79               corrido      0.763225           46              86.065217   
80      corridos bélicos      0.758600           39              87.358974   
81     corridos tumbados      0.756251           39              87.538462   
111     electro corridos      0.749708           28              88.571429   
232                  pop      0.704871           20              91.250000   
208      música mexicana      0.701013           39              86.769231   
97         dembow belico      0.696799           13              85.769231   
283             sierreño      0.680689           25              87.400000   
27                 banda      0.680004           42              85.285714   
275         sad sierreño      0.676418           29             

In [ ]:
# Fetch playlists matching "corrido"
corrido_playlists = sp.search(q='corrido', type='playlist', limit=10)['playlists']['items']

corrido_artists_data = []

for pl in tqdm(corrido_playlists, desc="Fetching tracks from playlists"):
    if pl is None:  # Skip None entries
        continue

    playlist_id = pl['id']
    playlist_tracks = sp.playlist_tracks(playlist_id)['items']

    for t in playlist_tracks:
        track = t.get('track')
        if not track:  # skip if track info missing
            continue
        artists = track.get('artists', [])
        if not artists:
            continue

        # take all artists in the track
        for artist in artists:
            corrido_artists_data.append({
                'artist_name': artist['name'],
                'artist_id': artist['id']
            })

# Remove duplicates
corrido_artists_df = pd.DataFrame(corrido_artists_data).drop_duplicates(subset='artist_name')
print(f"Found {len(corrido_artists_df)} unique artists")

Fetching tracks from playlists: 100%|██████████| 10/10 [00:04<00:00,  2.34it/s]

Found 233 unique artists


In [ ]:
from datetime import datetime, timedelta
import time

# Assume corrido_artists_df is already fetched from playlists
# It has columns: ['artist_name', 'artist_id']

three_months_ago = datetime.now() - timedelta(days=90)

# Initialize columns
corrido_artists_df['num_tracks_last_3mo'] = 0
corrido_artists_df['avg_track_popularity'] = 0.0

In [ ]:
for idx, row in corrido_artists_df.iterrows():
    artist_id = row['artist_id']
    try:
        albums = sp.artist_albums(artist_id, album_type='album', limit=20)['items']
    except Exception as e:
        print(f"Skipping artist {row['artist_name']} due to error: {e}")
        continue

    recent_tracks = []
    for album in albums:
        release_date = album.get('release_date')
        if not release_date:
            continue

        # Handle different date formats
        if len(release_date) == 4:
            release_date_dt = datetime.strptime(release_date, '%Y')
        elif len(release_date) == 7:
            release_date_dt = datetime.strptime(release_date, '%Y-%m')
        else:
            release_date_dt = datetime.strptime(release_date[:10], '%Y-%m-%d')

        if release_date_dt >= three_months_ago:
            album_id = album['id']
            try:
                tracks = sp.album_tracks(album_id)['items']
                recent_tracks.extend(tracks)
            except Exception as e:
                print(f"Skipping album {album['name']} due to error: {e}")
                continue

    # ---- Metrics Section (Properly Indented) ----
    corrido_artists_df.at[idx, 'num_tracks_last_3mo'] = len(recent_tracks)
    if recent_tracks:
        track_pops = []
        for track in recent_tracks:
            try:
                track_info = sp.track(track['id'])
                track_pops.append(track_info['popularity'])
                time.sleep(0.2)  # small delay per track
            except Exception:
                continue
        corrido_artists_df.at[idx, 'avg_track_popularity'] = (
            sum(track_pops) / len(track_pops) if track_pops else 0
        )
    else:
        corrido_artists_df.at[idx, 'avg_track_popularity'] = 0

    # Delay to avoid rate limit
    time.sleep(1)

In [ ]:
# Copy DataFrame
df = corrido_artists_df.copy()

# Normalize metrics
df['norm_recent_tracks'] = df['num_tracks_last_3mo'] / max(df['num_tracks_last_3mo'].max(), 1)
df['norm_track_pop'] = df['avg_track_popularity'] / 100

# Compute potential score using only available columns
df['potential_score'] = 0.5 * df['norm_recent_tracks'] + 0.5 * df['norm_track_pop']

# Sort and show top candidates
top_corrido_artists = df.sort_values('potential_score', ascending=False)
top_corrido_artists.head(10)

,artist_name,artist_id,num_tracks_last_3mo,avg_track_popularity,norm_recent_tracks,norm_track_pop,potential_score
192,Bobby Pulido,4EEZg8R3dxbTCCQ1DVWtHg,46,16.565217,1.000000,0.165652,0.582826
1,Victor Mendivil,5YqI7p8zYsOpKJtjxYdOce,18,74.277778,0.391304,0.742778,0.567041
13,Chino Pacas,2rmkQLzj0k4nZdQehOUByO,28,51.071429,0.608696,0.510714,0.559705
385,Beto Quintanilla,2L54fWLv9EgJu76ELlfV2J,42,16.166667,0.913043,0.161667,0.537355
2,Natanael Cano,0elWFr7TW8piilVRYJUe4P,16,68.812500,0.347826,0.688125,0.517976
15,Enigma Norteño,3441uYrkzgTWwjXLd13R0U,30,37.200000,0.652174,0.372000,0.512087
276,Chalino Sanchez,7u9m43vPVTERaALXXOzrRq,40,12.350000,0.869565,0.123500,0.496533
310,Tombochio,76yamFWr2gZGn03EaAbX2r,20,46.250000,0.434783,0.462500,0.448641
453,Eden Muñoz,1gJdf4Yybu4X5A2xYV3NMV,15,54.400000,0.326087,0.544000,0.435043
907,LOS DOS DE TAMAULIPAS,77Zc5MMUIMJriEDAcaDspi,10,60.700000,0.217391,0.607000,0.412196
